# Stage 2 — distillation + controlled LoRA sweep (Kaggle / Colab GPU)

Pipeline:
1. Run the **teacher** (`ibm-granite/granite-4.0-h-micro`, 3B) on the *train* task split with the harness and keep only the trajectories the harness marks correct → rejection-sampled, sequence-level distillation data.
2. LoRA-fine-tune the **student** (`ibm-granite/granite-4.0-350m`) on that data. Vary **rank × learning rate × seed**, hold everything else fixed.
3. Evaluate every run on the held-out *test* split (including OOD templates/documents), plot accuracy vs. rank with error bars over seeds.
4. Reproduce the LoRA paper's subspace-similarity analysis (Hu et al. 2021, §7.2) on our own adapters.

Expected runtime on a T4: teacher pass ~20 min, each sweep run ~5 min (train + 100-task eval).

In [ ]:
# --- setup -------------------------------------------------------------------
REPO = "https://github.com/Gangadhar017/SLM-agent-Lab.git"
import os, subprocess, sys
if not os.path.exists("SLM-agent-Lab"):
    subprocess.run(["git", "clone", REPO], check=True)
os.chdir("SLM-agent-Lab")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.57", "peft>=0.13", "accelerate", "pandas", "matplotlib", "tqdm"], check=True)
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
!python scripts/make_tasks.py

## 1. Teacher pass on the train split → distillation data

In [ ]:
TEACHER = "ibm-granite/granite-4.0-h-micro"      # 3B hybrid; fits a T4 in fp16
STUDENT = "ibm-granite/granite-4.0-350m"
!python scripts/run_eval.py --model {TEACHER} --tasks data/tasks/train.jsonl --dtype float16 --out results/runs/teacher_train
!python scripts/build_sft_data.py --source teacher --run results/runs/teacher_train --out data/sft/teacher_train.jsonl
!python scripts/build_sft_data.py --source oracle --out data/sft/oracle_train.jsonl   # comparison: perfect trajectories

## 2. Baselines on the test split (teacher and un-tuned student)

In [ ]:
!python scripts/run_eval.py --model {TEACHER} --dtype float16 --out results/runs/teacher_test
!python scripts/run_eval.py --model {STUDENT} --dtype float16 --out results/runs/student_baseline_test

## 3. Controlled sweep: rank × lr × seed, everything else fixed

Fixed: data, 2 epochs, batch 4, alpha = 2·rank, dropout 0.05, targets q/k/v/o, max_len 1024, AdamW, linear schedule with 5% warm-up.
The full grid (6 ranks × 3 lrs × 3 seeds = 54 runs ≈ 4.5 h on a T4) is below; the default cell runs the reduced grid (24 runs ≈ 2 h).

In [ ]:
RANKS, LRS, SEEDS = "1,4,16,64", "1e-4,5e-4", "0,1,2"          # reduced grid
# RANKS, LRS, SEEDS = "1,2,4,8,16,64", "1e-4,2e-4,5e-4", "0,1,2" # full grid
!python scripts/lora_sweep.py --model {STUDENT} --data data/sft/teacher_train.jsonl --ranks {RANKS} --lrs {LRS} --seeds {SEEDS} --epochs 2 --eval-tasks 100 --out results/sweep/student_teacher_data

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, json
df = pd.read_csv("results/sweep/student_teacher_data/sweep.csv")
base = json.load(open("results/runs/student_baseline_test/meta.json"))
from slm_agent_lab.eval.report import load_runs, summarize
acc_base = next(iter(summarize(load_runs(["results/runs/student_baseline_test"]))["models"].values()))["accuracy"]
acc_teacher = next(iter(summarize(load_runs(["results/runs/teacher_test"]))["models"].values()))["accuracy"]
g = df.groupby(["rank", "lr"])["eval_accuracy"].agg(["mean", "std", "count"]).reset_index()
fig, ax = plt.subplots(figsize=(7, 4.5))
for lr, sub in g.groupby("lr"):
    ax.errorbar(sub["rank"], sub["mean"], yerr=sub["std"], marker="o", capsize=3, label=f"lr={lr:g}")
ax.axhline(acc_base, ls="--", c="gray", label=f"student baseline ({acc_base:.0%})")
ax.axhline(acc_teacher, ls=":", c="black", label=f"teacher ({acc_teacher:.0%})")
ax.set_xscale("log", base=2); ax.set_xlabel("LoRA rank r"); ax.set_ylabel("test accuracy (100 tasks)"); ax.set_ylim(0, 1)
ax.set_title("Task accuracy vs. LoRA rank (mean ± std over seeds)"); ax.legend(fontsize=8); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig("docs/figures/lora_rank_sweep.png", dpi=150)
g

## 4. LoRA paper §7.2 reproduction: subspace similarity between ranks and between seeds

Claim: the top singular directions learned at r=4 are contained in those learned at r=64 (φ well above the random-matrix baseline), while the remaining directions of the larger adapter are mostly noise.

In [ ]:
S = "results/sweep/student_teacher_data"
!python scripts/subspace_analysis.py {S}/r4_lr0.0001_s0 {S}/r64_lr0.0001_s0 --module q_proj --out docs/figures/subspace_r4_vs_r64_q.png
!python scripts/subspace_analysis.py {S}/r64_lr0.0001_s0 {S}/r64_lr0.0001_s1 --module q_proj --out docs/figures/subspace_r64_seed0_vs_seed1_q.png
from IPython.display import Image, display
display(Image("docs/figures/subspace_r4_vs_r64_q.png")); display(Image("docs/figures/subspace_r64_seed0_vs_seed1_q.png"))

## 5. Failure taxonomy: baseline student vs. best distilled student vs. teacher

In [ ]:
best = df.sort_values("eval_accuracy", ascending=False).iloc[0]
best_dir = f"{S}/r{int(best['rank'])}_lr{best['lr']:g}_s{int(best['seed'])}"
print("best run:", best_dir, best["eval_accuracy"])
!python scripts/run_eval.py --model {STUDENT} --adapter {best_dir} --dtype float16 --out results/runs/student_distilled_test
!python scripts/make_report.py results/runs/student_baseline_test results/runs/student_distilled_test results/runs/teacher_test --out results/report_stage2
display(Image("docs/figures/failure_taxonomy.png")); display(Image("docs/figures/accuracy_by_category.png"))